In [1]:
from google.colab import files
uploaded = files.upload()

Saving Test.csv to Test.csv


In [2]:
from google.colab import files
upload = files.upload()

Saving Train.csv to Train.csv


In [3]:
from google.colab import files
upload = files.upload()

Saving SampleSubmission.csv to SampleSubmission.csv


In [5]:
import pandas as pd, numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import mean_absolute_error, roc_auc_score


In [6]:
train = pd.read_csv("Train.csv")
test  = pd.read_csv("Test.csv")
sample = pd.read_csv("SampleSubmission.csv")

print(train.shape, test.shape)
print(train.head())
print(train.isna().sum())               # missing values?
print(train.bank_account.value_counts()) # class balance

(23524, 13) (10086, 12)
  country  year    uniqueid bank_account location_type cellphone_access  \
0   Kenya  2018  uniqueid_1          Yes         Rural              Yes   
1   Kenya  2018  uniqueid_2           No         Rural               No   
2   Kenya  2018  uniqueid_3          Yes         Urban              Yes   
3   Kenya  2018  uniqueid_4           No         Rural              Yes   
4   Kenya  2018  uniqueid_5           No         Urban               No   

   household_size  age_of_respondent gender_of_respondent  \
0               3                 24               Female   
1               5                 70               Female   
2               5                 26                 Male   
3               5                 34               Female   
4               8                 26                 Male   

  relationship_with_head           marital_status  \
0                 Spouse  Married/Living together   
1      Head of Household                  Widowed   

In [7]:
train["unique_id"] = train.uniqueid + " x " + train.country
test["unique_id"]  = test.uniqueid  + " x " + test.country

In [8]:
print(test.unique_id.isin(sample.unique_id).mean())   # should be 1.0

1.0


In [9]:
y = (train.bank_account == "Yes").astype(int)

In [10]:
def add_features(d):
    d = d.copy()
    d["age_group"] = pd.cut(d.age_of_respondent, [0,24,34,44,54,64,120], labels=False)
    d["hh_per_age"] = d.household_size / d.age_of_respondent
    d["country_year"] = d.country + "_" + d.year.astype(str)
    return d

train = add_features(train)
test  = add_features(test)

In [11]:
cats = ["country","year","location_type","cellphone_access","gender_of_respondent",
        "relationship_with_head","marital_status","education_level","job_type","country_year"]
nums = ["household_size","age_of_respondent","age_group","hh_per_age"]

X  = train[cats + nums].copy()
Xt = test[cats + nums].copy()
X["year"] = X["year"].astype(str)      # treat year as a category, not a number
Xt["year"] = Xt["year"].astype(str)

In [12]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

encode = ColumnTransformer(
    [("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cats)],
    remainder="passthrough")   # leave numeric columns as they are

In [13]:
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

models = {
  "logreg": make_pipeline(
      ColumnTransformer([("c", OneHotEncoder(handle_unknown="ignore"), cats),
                         ("n", StandardScaler(), nums)]),
      LogisticRegression(max_iter=2000)),
  "rf": make_pipeline(encode, RandomForestClassifier(500, min_samples_leaf=5,
                                                     n_jobs=-1, random_state=0)),
  "hgb": make_pipeline(encode, HistGradientBoostingClassifier(
                       learning_rate=0.05, max_iter=300, max_leaf_nodes=15,
                       l2_regularization=1.0, random_state=0)),
}

In [14]:
skf = StratifiedKFold(5, shuffle=True, random_state=42)  # stratified keeps the 14% ratio in each fold

oof = {}
for name, m in models.items():
    oof[name] = cross_val_predict(m, X, y, cv=skf, method="predict_proba")[:, 1]
    print(name, "AUC:", round(roc_auc_score(y, oof[name]), 4),
                "MAE:", round(mean_absolute_error(y, oof[name] > 0.5), 4))

logreg AUC: 0.8521 MAE: 0.1155
rf AUC: 0.8623 MAE: 0.1121
hgb AUC: 0.8641 MAE: 0.1115


In [15]:
ens = np.mean(list(oof.values()), axis=0)     # average of the 3 models
print("ensemble AUC:", roc_auc_score(y, ens))

for t in np.arange(0.3, 0.71, 0.04):
    print(round(t, 2), mean_absolute_error(y, ens > t))

ensemble AUC: 0.8653432955478688
0.3 0.1255313722156096
0.34 0.12013263050501616
0.38 0.11494643768066655
0.42 0.11286345859547696
0.46 0.11218330215949668
0.5 0.11175820438700901
0.54 0.11133310661452134
0.58 0.11328855636796463
0.62 0.11566910389389559
0.66 0.11694439721135862
0.7 0.11970753273252849


In [16]:
test_probs = np.mean(
    [m.fit(X, y).predict_proba(Xt)[:, 1] for m in models.values()], axis=0)

test["bank_account"] = (test_probs > 0.5).astype(int)

In [17]:
sub = sample[["unique_id"]].merge(test[["unique_id", "bank_account"]],
                                  on="unique_id", how="left")

# The sample file also lists train IDs; fill them with the known labels
labels = dict(zip(train.unique_id, y))
sub["bank_account"] = sub.bank_account.fillna(sub.unique_id.map(labels)).astype(int)

sub.to_csv("submission.csv", index=False)
print(sub.shape, sub.columns.tolist())

(33610, 2) ['unique_id', 'bank_account']


In [18]:
print(sub.isna().sum())                  # no missing values
print(sub.bank_account.value_counts())   # only 0 and 1
print(sub.shape)                         # should be (33610, 2)
print(sub.columns.tolist())              # ['unique_id', 'bank_account']
print(sub.head())

unique_id       0
bank_account    0
dtype: int64
bank_account
0    29625
1     3985
Name: count, dtype: int64
(33610, 2)
['unique_id', 'bank_account']
            unique_id  bank_account
0  uniqueid_1 x Kenya             1
1  uniqueid_2 x Kenya             0
2  uniqueid_3 x Kenya             1
3  uniqueid_4 x Kenya             0
4  uniqueid_5 x Kenya             0


In [19]:
print(set(sub.unique_id) == set(sample.unique_id))   # should be True

True


In [20]:
from google.colab import files
files.download("submission.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>